# QSV2G — Stage 05 (E1–E5)

**Software-based experiment orchestration; not physical quantum-hardware validation.** E1–E4 use real ML-KEM/ML-DSA backends for the protocol endpoints; E2 generated stateful tests separately use a labeled test-only cryptography stub; E5 requires an installed Tamarin binary. Read `experiments/stage05/README.md`.

Set `SMOKE=True` initially. After checking pilot results and frozen configuration, set `SMOKE=False`. The exact commit must be pinned and recorded before a final paper run. Each execution saves to a new run-ID directory, so smoke and final outputs are not accidentally combined. E5 will remain UNRESOLVED if the Tamarin executable is missing.

## 00. Settings and pinned Git revision

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, zipfile, shutil
SMOKE = True  # Change to False only after all pilot gates pass
PINNED_COMMIT = "3f930e1091aa6b84e83712dd0f579fcf3a8218c2"  # Provided in the final GitHub delivery report
from datetime import datetime, timezone
ROOT=Path("/kaggle/working/QSV2G") if Path("/kaggle/working").exists() else Path.cwd()/"QSV2G_kaggle"
OUT_BASE=Path("/kaggle/working/qsv2g_stage05_results") if Path("/kaggle/working").exists() else Path.cwd()/"qsv2g_stage05_results"
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT=OUT_BASE/("smoke" if SMOKE else "final")/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
print("Fresh output directory:",OUT)
assert len(PINNED_COMMIT)==40 and all(x in "0123456789abcdef" for x in PINNED_COMMIT), "Set PINNED_COMMIT to the reported 40-character immutable Stage 05 source commit"

## 01. Clone the approved source commit and record environment

In [ ]:
if not (ROOT/".git").exists():
    subprocess.run(["git","clone","--quiet","https://github.com/shanmukhdatta/QSV2G.git",str(ROOT)],check=True)
subprocess.run(["git","-C",str(ROOT),"fetch","--quiet","origin",PINNED_COMMIT],check=True)
subprocess.run(["git","-C",str(ROOT),"checkout","--quiet","--detach",PINNED_COMMIT],check=True)
head=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
assert head==PINNED_COMMIT
print("Source pinned:",head)

## 02. Dependencies and real-cryptography preflight

In [ ]:
import importlib.util
missing=[x for x in ["numpy","pandas","matplotlib","cryptography","pytest","hypothesis"] if importlib.util.find_spec(x) is None]
if missing:
    subprocess.run([sys.executable,"-m","pip","install", "--quiet", *missing],check=True)
sys.path.insert(0,str(ROOT/"src"))
from qsv2g_core import PQCrypto
try:
    backend=PQCrypto()
except RuntimeError:
    # Kaggle's system OpenSSL may be older; explicitly install real native PQC.
    subprocess.run([sys.executable,"-m","pip","install","--quiet","pqcrypto>=0.4"],check=True)
    backend=PQCrypto()  # Must still succeed with genuine ML-KEM/ML-DSA
print("Real cryptographic backend:",backend.kind)
(OUT/"git_revision.txt").write_text(head+"\n")
(OUT/"pip_freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))

## 03. P0: regression test gate

In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q","tests"],cwd=ROOT,check=True)

## 04. E1 — matched authorization and controlled software cost

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e1","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)

## 05. E2 — real-PQC adversarial tests and generated stateful sequences

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e2","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)
subprocess.run([sys.executable,"experiments/stage05/stateful_hypothesis.py",
    "--examples",str(20 if SMOKE else 1000),"--steps",str(15 if SMOKE else 50),
    "--output",str(OUT/"e2"/"hypothesis.json")],cwd=ROOT,check=True)

## 06. E3 — full-vector commitment gate and real token checks

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e3","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)

## 07. E4 — authority outages and contention

In [ ]:
cmd=[sys.executable,"experiments/stage05/runner.py","--experiment","e4","--out",str(OUT)]
if SMOKE:cmd.append("--smoke")
subprocess.run(cmd,cwd=ROOT,check=True)

## 08. E5 — Tamarin attempted formal verification (never claim unresolved proofs)

In [ ]:
subprocess.run([sys.executable,"experiments/stage05/run_tamarin.py","--out",str(OUT/"e5"),"--timeout","180"],cwd=ROOT,check=True)
print((OUT/"e5"/"e5_status.json").read_text())

## 09. Results and figures (recomputed from Stage 05 raw outputs)

In [ ]:
subprocess.run([sys.executable,"experiments/stage05/analyze.py",str(OUT)],cwd=ROOT,check=True)
print((OUT/"analysis_summary.json").read_text())

## 10. Archive all results with a per-file manifest

In [ ]:
import hashlib
files=[]
for path in sorted(OUT.rglob("*")):
    if path.is_file() and path.name!="artifact_manifest.json":
        files.append({"path":str(path.relative_to(OUT)),"bytes":path.stat().st_size,"sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
(OUT/"artifact_manifest.json").write_text(json.dumps({"git_sha":head,"smoke":SMOKE,"files":files},indent=2))
archive=OUT.parent/(f"qsv2g_stage05_{'smoke' if SMOKE else 'final'}_{RUN_ID}.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as zipf:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():zipf.write(path,path.relative_to(OUT.parent))
print("Results archive:",archive)